# **Morphological operations**

<div style="color:#777777;margin-top: -15px;">
<b>Author</b>: Norman Juchler |
<b>Course</b>: MSLS CO4 |
<b>Version</b>: v1.1 <br><br>
<!-- Date: 06.03.2025 -->
<!-- Comments: Language refactored. -->
</div>

In this notebook, we explore the effects of morphological operations on images.

Morphological operations are fundamental image processing techniques that process images based on their shapes. These operations require two inputs:

Morphological operations are basic image operations that process images based on shapes. A morphological operation 
requires two inputs: an input image and a structuring element. The structuring element is a small, usually binary image that specifies 
the neighborhood used to process each pixel in the input image. It represents the shape of the operation being applied.
The illustration below shows common structuring elments. 

![Common structuring elements](../data/doc/structuring-elements-2d.svg)  
**Figure 1**: Common structuring elements.

Morphological operations resemble convolution, where the structuring element is swept across the image. However, unlike convolution (which involves multiplication and summation), morphological operations rely on *set theory* and *logical operations* (e.g., intersection, union, complement).

**Erosion**, for instance, shrinks foreground objects by computing the **intersection** of the image and the shifted structuring element. A pixel remains foreground (value=1) in the output only if all foreground pixels in the structuring element overlap with foreground pixels in the input image. See Figure 2.

Likewise, **dilation** expands foreground objects by computing the **union** of the image and the shifted structuring element. A pixel is set to foreground (value=1) in the output if at least one foreground pixel in the structuring element overlaps with a foreground pixel in the input image. See Figure 3.

<br>

![Binary erosion](../data/doc/binary-erosion.svg)  
**Figure 2**: Binary erosion of an input image (1) using a circular structuring element (2). Subfigures (3) and (4) show the result.

![Binary dilation](../data/doc/binary-dilation.svg)  
**Figure 2**: Binary dilation of an input image (1) using a circular structuring element (2). Subfigures (3) and (4) show the result.


<br><br>

Morphological operations are used for a variety of image processing tasks, such as noise removal, object isolation, and feature enhancement. Here, we will explore the following elementary morphological operations: 
* Dilation
* Erosion
* Opening (erosion followed by dilation)
* Closing (dilation followed by erosion)

Rather than defining these operations formally, we will study their effects through examples.

Credits: This notebook follows a tutorial from pyimagesearch, written by Adrian Rosebrock. [Link](https://pyimagesearch.com/2021/04/28/opencv-morphological-operations/)

---

## **Preparations**

The usual preparations... Before we begin, let's load some drawing functions for rendering images effortlessly in this Jupyter notebook.

In [ ]:
import sys
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt

# Enable vectorized output (for nicer plots)
%config InlineBackend.figure_formats = ["svg"]

# Inline backend configuration
%matplotlib inline

# Functionality related to this course
sys.path.append("..")
import tools as isp

# Jupyter / IPython configuration:
# Automatically reload modules when modified
%load_ext autoreload
%autoreload 2

---

## **Examplary image data**

In the following, we use a binary image containing text and introduce artificial noise to the image.

Recall that a binary image is an image where each pixel has only two possible values: Depending on the image type, these values are often 0 and 1 (`dtype=bool`), or 0 and 255 (`dtype=np.uint8`). Since OpenCV’s morphological operations expect images of type `np.uint8`, we go with the latter option.

**Note**: Although morphological operations can be applied to grayscale and even color images, we will focus on binary images in this example.


In [ ]:
def sample_border(w, h, margin):
    while True:
        x = np.random.randint(0, w)
        y = np.random.randint(0, h)
        if (x < margin) or (x >= (w - margin)) or (y < margin) or (y >= h - margin):
            return x, y

# Read in the image and invert it
img = cv.imread("../data/images/word-ice-cream.png", cv.IMREAD_GRAYSCALE)
img = 255 - img

# Add dots of noise such that they do not overlap with the text.
h, w = img.shape
img_noisy = img.copy()
np.random.seed(1)
for i in range(50):
    x, y = sample_border(w, h, 20)
    r = np.random.randint(1, 4)
    cv.circle(img_noisy, (x, y), r, 255, -1, lineType=cv.LINE_AA)

# Display the image
isp.show_image_chain([img, img_noisy], 
                     titles=["Input image", "Noisy image"]);

---

## **Erosion**

Just like the name suggests, erosion shrinks or erodes the foreground regions of an image. It is particularly useful for removing small noise or unwanted blobs, or separating connected objects.

Erosion requires a structuring element, which slides over the image from left to right and top to bottom. A foreground pixel in the output image is retained only if all corresponding pixels in the structuring element are greater than zero. Otherwise, the pixel is set to 0 (background).


In [ ]:
# Note: We can specify our own structuring element (SE), 
# but we can also use the default one (3x3 square).
#se = np.ones((5, 5), np.uint8)
se = None

results = {}
results["Original"] = img_noisy
for i in range(1, 4):
    result = cv.erode(img_noisy, se, iterations=i)
    results[f"Eroded ({i} iterations)"] = result
    
isp.show_image_grid(results, suppress_info=True, ncols=1, figsize=(6, 7));
#isp.save_figure(path="morpho-erosion.png")

---

## **Dilation**

Dilation is the opposite of erosion: it expands the boundaries of foreground objects in an image. It is particularly useful for 
- Filling gaps and connecting broken parts of objects.
- Enhancing prominent features.
- Emphasizing foreground structures.

Dilation also utilize a structuring element (kernel), which slides over the image. A pixel in the output image is set to white (foreground) if at least one corresponding pixel in the structuring element is greater than zero.

In [ ]:
se = None  # Default structuring element, but you can specify your own
results = {}
results["Original"] = img
for i in range(1, 4):
    results[f"Dilated ({i} iterations)"] = cv.dilate(img, se, iterations=i)
    
isp.show_image_grid(results, suppress_info=True, ncols=1, figsize=(6, 7));
#isp.save_figure(path="morpho-dilation.png")

Erosion and dilation are **dual operations**: applying erosion to the inverted image produces the same result as applying dilation to the original image (and vice versa). The following code snippet demonstrates this relationship:

In [ ]:
se = np.ones((5, 5), np.uint8)
img_dilated = cv.dilate(img, se, iterations=1)
img_eroded = 255 - cv.erode(255-img, se, iterations=1)
img_diff = img_dilated - img_eroded
print("Test: The difference between dilated image and eroded complement of the image should be zero:")
print("      Difference = %d" % np.abs(img_diff).sum())

---

## **Opening**

Opening is a morphological operation that consists of erosion followed by dilation.

This technique is particularly useful for removing small noise or unwanted blobs while preserving the shape of larger objects. The erosion step eliminates small foreground artifacts, and the subsequent dilation restores the main structures without reintroducing noise.

In the example below, we define a custom structuring element using OpenCV’s function:

```text
cv.getStructuringElement(type, size)
```
where `type` specifies the shape of the structuring element (rectangle: `cv.MORPH_RECT`, cross: `cv.MORPH_CROSS`, ellipse: `cv.MORPH_ELLIPSE`), and  `size` defines the dimensions of the structuring element. 

Recall: A structuring element functions similarly to a convolution kernel, but instead of performing weighted sums (as in convolution), it is used for logical operations in morphological transformations.

In [ ]:
sizes = [(3, 3), (5, 5), (7, 7)]
results = {}
results["Original"] = img_noisy
for size in sizes:
    # Construct a rectangular structuring elmenet from the current size and then
    # apply an "opening" operation
    se = cv.getStructuringElement(cv.MORPH_RECT, size)
    ret = cv.morphologyEx(img_noisy, cv.MORPH_OPEN, se, iterations=1)
    results["Opening (structuring element: %dx%d)" % size] = ret
    
isp.show_image_grid(results, suppress_info=True, ncols=1, figsize=(6, 7));
#isp.save_figure(path="morpho-opening.png")

---

## **Closing**

Closing is the inverse of opening. It consists of dilation followed by erosion.

As the name suggests, a closing is used to fill small holes inside forground objects or to connect broken components. We can reuse the same code and just need to change the operation type accordingly.


In [ ]:
sizes = [(3, 3), (5, 5), (7, 7)]
results = {}
results["Original"] = img
for size in sizes:
    # Construct a rectangular structuring element from the current size and then
    # apply an "opening" operation
    se = cv.getStructuringElement(cv.MORPH_RECT, size)
    ret = cv.morphologyEx(img, cv.MORPH_CLOSE, se, iterations=1)
    results["Closing (structuring element: %dx%d)" % size] = ret
    
isp.show_image_grid(results, suppress_info=True, ncols=1, figsize=(6, 7));
#isp.save_figure(path="morpho-closing.png")

---

## **Morphological gradient**

A morphological gradient is the difference between a dilated and eroded version of an image. It is useful for determining the outline of objects by emphasizing transitions between foreground and background.

Again, we can compute the morphological gradient using the same code as above, simply by changing the operation type.

In [ ]:
sizes = [(3, 3), (5, 5), (7, 7)]
results = {}
results["Original"] = img
for size in sizes:
    # Construct a rectangular structuring element from the current size and then
    # apply an "opening" operation
    se = cv.getStructuringElement(cv.MORPH_RECT, size)
    ret = cv.morphologyEx(img, cv.MORPH_GRADIENT, se, iterations=1)
    results["Gradient (structuring element: %dx%d)" % size] = ret
    
isp.show_image_grid(results, suppress_info=True, ncols=1, figsize=(6, 7));
#isp.save_figure(path="morpho-closing.png")

---

## **Morphological operations and grayscale images**

So far, we have applied morphological operations only to binary images. However, these operations are also defined for grayscale (and even color) images.

To explore this, let's examine an MRI scan of the brain.  (Source: [Radiopaedia](https://radiopaedia.org/cases/normal-brain-mri-6))


In [ ]:
img = cv.imread("../data/images/brain-mri/brain-mri-a-t2-dark.jpg", cv.IMREAD_GRAYSCALE)

size = (3, 3)
se = cv.getStructuringElement(cv.MORPH_RECT, size)
erode = cv.morphologyEx(img, cv.MORPH_ERODE, se, iterations=5)
dilate = cv.morphologyEx(img, cv.MORPH_DILATE, se, iterations=5)
opening = cv.morphologyEx(img, cv.MORPH_OPEN, se, iterations=5)
closing = cv.morphologyEx(img, cv.MORPH_CLOSE, se, iterations=5)
gradient = cv.morphologyEx(img, cv.MORPH_GRADIENT, se, iterations=3)

size = (11, 11)
se = cv.getStructuringElement(cv.MORPH_RECT, size)
tophat = cv.morphologyEx(img, cv.MORPH_TOPHAT, se, iterations=1)

size = (25, 25)
se = cv.getStructuringElement(cv.MORPH_RECT, size)
blackhat = cv.morphologyEx(img, cv.MORPH_BLACKHAT, se, iterations=1)


results = {
    "Original": img,
    "Erosion": erode,
    "Dilation": dilate,
    "Opening": opening,
    "Closing": closing,
    "Gradient": gradient,
    "Top-hat": tophat,
    "Black-hat": blackhat
}

isp.show_image_grid(results, suppress_info=True, ncols=3, figsize=(9, 7));
#isp.save_figure(path="morpho-closing-grayscale.png")

## **Top-hat (white-hat) and bottom-hat (black-hat)**

The top-hat (white-hat) and bottom-hat (black-hat) operators are typically more effective for grayscale images rather than binary ones.

A top-hat (also known as white-hat) morphological operation is the difference between the original grayscale (or single-channel) input image and its opening.

The top-hat operation is used to highlight bright regions in an image with dark backgrounds, while the bottom-hat operation reveals dark regions on bright backgrounds.

